In [1]:
from pathlib import Path
import json
import shutil
import joblib
from datetime import datetime

In [2]:
BASE_DIR = Path.cwd().parent

MODEL_FILE = BASE_DIR / "models" / "aqi_model.pkl"
FEATURE_SCHEMA_FILE = BASE_DIR / "models" / "feature_columns.pkl"
METADATA_FILE = BASE_DIR / "models" / "model_metadata.json"

REGISTRY_DIR = BASE_DIR / "model_registry"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_DIR)
print("Registry directory:", REGISTRY_DIR)

Project root: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor
Registry directory: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\model_registry


In [3]:
if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_FILE}")

if not FEATURE_SCHEMA_FILE.exists():
    raise FileNotFoundError(f"Feature schema not found: {FEATURE_SCHEMA_FILE}")

model = joblib.load(MODEL_FILE)
feature_columns = joblib.load(FEATURE_SCHEMA_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    training_metadata = json.load(f)

print("Model loaded:", type(model).__name__)
print("Feature count:", len(feature_columns))

Model loaded: RandomForestRegressor
Feature count: 16


In [4]:
MODEL_VERSION = "v1"

REGISTERED_MODEL = REGISTRY_DIR / f"aqi_model_{MODEL_VERSION}.pkl"
REGISTERED_METADATA = REGISTRY_DIR / f"aqi_model_{MODEL_VERSION}_metadata.json"

shutil.copy2(MODEL_FILE, REGISTERED_MODEL)

registry_metadata = {
    "model_name": "Pearls AQI Predictor",
    "version": MODEL_VERSION,
    "model_type": type(model).__name__,
    "registered_at": datetime.now().isoformat(),
    "artifact": str(REGISTERED_MODEL.relative_to(BASE_DIR)),
    "feature_count": len(feature_columns),
    "feature_columns": list(feature_columns),
    "training_metadata": training_metadata
}

with open(REGISTERED_METADATA, "w", encoding="utf-8") as f:
    json.dump(registry_metadata, f, indent=4)

print("Model registered successfully.")
print("Version:", MODEL_VERSION)
print("Artifact:", REGISTERED_MODEL)

Model registered successfully.
Version: v1
Artifact: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\model_registry\aqi_model_v1.pkl


In [5]:
registered_model = joblib.load(REGISTERED_MODEL)

with open(REGISTERED_METADATA, "r", encoding="utf-8") as f:
    registered_info = json.load(f)

print("Registry verification successful.")
print("Registered model:", type(registered_model).__name__)
print("Version:", registered_info["version"])
print("Features:", registered_info["feature_count"])
print("Artifact exists:", REGISTERED_MODEL.exists())

Registry verification successful.
Registered model: RandomForestRegressor
Version: v1
Features: 16
Artifact exists: True


In [6]:
print("Registered Feature Schema:")
for i, feature in enumerate(registered_info["feature_columns"], start=1):
    print(f"{i:02d}. {feature}")

Registered Feature Schema:
01. CO
02. NO
03. NO2
04. O3
05. SO2
06. PM2_5
07. PM10
08. NH3
09. hour
10. day
11. month
12. weekday
13. AQI
14. AQI_lag_1
15. AQI_change
16. AQI_rolling_avg


In [7]:
print("\nModel Registry Summary")
print("=" * 40)
print("Model Name :", registered_info["model_name"])
print("Version    :", registered_info["version"])
print("Model Type :", registered_info["model_type"])
print("Features   :", registered_info["feature_count"])
print("Artifact   :", registered_info["artifact"])
print("Registered :", registered_info["registered_at"])


Model Registry Summary
Model Name : Pearls AQI Predictor
Version    : v1
Model Type : RandomForestRegressor
Features   : 16
Artifact   : model_registry\aqi_model_v1.pkl
Registered : 2026-09-04T23:28:09.017471
